# RSSI CNN-LSTM Model Training (GPU Optimized)
This notebook allows you to load, preprocess, train, and evaluate a CNN-LSTM model on WiFi RSSI data.
**Optimized for GPU (e.g., Vast.ai)** with mixed precision and TensorBoard logging.

In [2]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, Input
from tensorflow.keras.mixed_precision import set_global_policy

# Check GPU availability
print(tf.config.list_physical_devices('GPU'))
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"[INFO] Using GPU: {[gpu.name for gpu in gpus]}")
else:
    print("[WARNING] No GPU detected. Using CPU fallback.")

# Enable Mixed Precision and XLA
set_global_policy('mixed_float16')
tf.config.optimizer.set_jit(True)

2025-07-05 07:14:45.888917: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-05 07:14:45.899922: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-07-05 07:14:45.916107: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-07-05 07:14:45.916144: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-07-05 07:14:45.927580: I tensorflow/core/platform/cpu_feature_gua

[]
[WARNING] No GPU detected. Using CPU fallback.


2025-07-05 07:14:47.686352: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2251] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [2]:
# Configuration
DATA_DIR = "./data"
PCA_COMPONENTS = 35
TEST_SIZE = 0.4
EPOCHS = 50
BATCH_SIZE = 16
USE_PCA = True

In [3]:
def load_rssi_data():
    files = glob.glob(os.path.join(DATA_DIR, "wifisignal_data_*.csv"))
    data, labels = [], []
    if not files:
        print(f"[ERROR] No CSV files found in {DATA_DIR}.")
        return None, None
    for f in files:
        try:
            df = pd.read_csv(f)
            pkt_cols = [c for c in df.columns if c.startswith("pkt")]
            if not pkt_cols:
                print(f"[WARNING] No 'pkt' columns in {f}. Skipping.")
                continue
            X = df[pkt_cols].values.astype(float)
            y = df["label"].values
            data.append(X)
            labels.append(y)
        except Exception as e:
            print(f"[ERROR] Skipped {f}: {e}")
    if not data:
        return None, None
    return np.vstack(data), np.hstack(labels)

In [ ]:
def build_cnn_lstm(input_shape, num_classes):
    input_tensor = Input(shape=input_shape)
    x = layers.Reshape((input_shape[0], 1))(input_tensor)
    x = layers.Conv1D(64, 3, activation='relu')(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Dropout(0.5)(x)
    x = layers.LSTM(64)(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    output_tensor = layers.Dense(num_classes, activation='softmax', dtype='float32')(x)
    model = models.Model(inputs=input_tensor, outputs=output_tensor)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

In [ ]:
# Load and preprocess data
X, y = load_rssi_data()
if X is None:
    raise ValueError("No data loaded")

le = LabelEncoder()
y_enc = le.fit_transform(y)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=TEST_SIZE, random_state=42, stratify=y_enc)

scaler = StandardScaler()
if USE_PCA:
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    pca = PCA(n_components=PCA_COMPONENTS)
    X_train_processed = pca.fit_transform(X_train_scaled)
    X_test_processed = pca.transform(X_test_scaled)
    input_shape = (PCA_COMPONENTS,)
else:
    X_train_processed = scaler.fit_transform(X_train)
    X_test_processed = scaler.transform(X_test)
    input_shape = (X.shape[1],)

In [ ]:
train_dataset = tf.data.Dataset.from_tensor_slices((X_train_processed, y_train))
val_dataset = tf.data.Dataset.from_tensor_slices((X_test_processed, y_test))
train_dataset = train_dataset.shuffle(1024).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_dataset = val_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

model = build_cnn_lstm(input_shape, num_classes=len(le.classes_))
model.summary()

cb = [
    callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, min_lr=1e-6),
    callbacks.ModelCheckpoint('best_model.keras', monitor='val_accuracy', save_best_only=True),
    callbacks.TensorBoard(log_dir='./logs')
]

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS,
    callbacks=cb,
    verbose=2
)

In [ ]:
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.legend(); plt.grid(True); plt.title('Loss')

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.legend(); plt.grid(True); plt.title('Accuracy')
plt.tight_layout()
plt.savefig('training_history.png')

In [ ]:
best_model = models.load_model('best_model.keras')
loss, acc = best_model.evaluate(val_dataset)
print(f'Test Loss: {loss:.4f}, Test Accuracy: {acc:.4f}')

preds = np.argmax(best_model.predict(val_dataset), axis=1)
y_true = np.concatenate([y for _, y in val_dataset], axis=0)
cm = confusion_matrix(y_true, preds)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Confusion Matrix')
plt.savefig('confusion_matrix.png')

report = classification_report(y_true, preds, target_names=le.classes_)
print(report)
with open('classification_report.txt', 'w') as f:
    f.write(report)